# Sistema Híbrido de Inteligencia Artificial Explicable para Clasificar el Potencial de Rendimiento de Contenidos Publicitarios Digitales

## y Generar Recomendaciones de Priorización, Ajuste o Potenciación

---

> **Universidad de Especialidades Espíritu Santo — Maestría en Inteligencia Artificial**  
> **Autores:** Santiago Salazar Ruiz · Jesús Zambrano Parrales  
> **Docente:** Alexandra Jacqueline Arciniegas Coral  

---

## Pregunta de investigación

¿En qué medida un sistema híbrido de inteligencia artificial, basado en aprendizaje supervisado, procesamiento de lenguaje natural, análisis temporal, explicabilidad algorítmica y recomendación asistida por modelos de lenguaje, puede clasificar el potencial de rendimiento de contenidos publicitarios digitales y sugerir qué anuncios deberían potenciarse, ajustarse o revisarse antes de escalar su difusión?

## Objetivo general

Diseñar y evaluar un sistema híbrido de inteligencia artificial explicable para clasificar el potencial de rendimiento de contenidos publicitarios digitales y generar recomendaciones de priorización, ajuste o potenciación, mediante aprendizaje supervisado, procesamiento de lenguaje natural, clustering semántico, explicabilidad algorítmica y apoyo generativo con modelos de lenguaje.

## Nota metodológica

El sistema opera bajo el principio **Human-in-the-Loop**: estima el potencial relativo de rendimiento de un contenido y apoya la decisión humana. No garantiza viralidad, no automatiza presupuesto ni toma decisiones comerciales irreversibles. El analista humano retiene siempre la decisión final.

## Arquitectura del sistema (10 capas)

| # | Capa | Técnica principal | Modo |
|---|------|-------------------|------|
| 1 | Ingesta y preparación | Pandas, limpieza, conversión | Obligatorio |
| 2 | Variable objetivo | Composite score normalizado → 3 niveles / binario | Obligatorio |
| 3 | Feature engineering | 22 variables lingüísticas, temporales y de engagement | Obligatorio |
| 4 | NLP y clustering semántico | TF-IDF + Sentence-BERT + KMeans | Obligatorio |
| 5 | Modelado supervisado | Dummy, LogReg, RF, SVC, XGBoost, LightGBM | Obligatorio |
| 6 | Explicabilidad (XAI) | SHAP global + local + Permutation importance | Obligatorio |
| 7 | Fairness y robustez | PSI, ruido, subgrupos, análisis temporal | Obligatorio |
| 8 | Recomendación creativa | Reglas + score + factores SHAP | Obligatorio |
| 9 | LLM como apoyo | Phi-3-mini prompt estructurado (si GPU disponible) | Opcional |
| 10 | Demo integrador | Notebook autocontenido ejecutable de extremo a extremo | Obligatorio |

**Tiempo estimado de ejecución completa:** 20–30 minutos en Colab Pro con GPU T4.

## Celda 1 — Instalación de dependencias

Ejecuta una sola vez al inicio. Tiempo: ~3 minutos.

In [ ]:
import subprocess, sys
pkgs = [
    "xgboost>=2.0", "lightgbm>=4.0",
    "scikit-learn>=1.4,<1.6", "imbalanced-learn>=0.12",
    "shap>=0.46", "sentence-transformers>=3.0",
    "pandas>=2.0", "numpy>=1.24",
    "matplotlib>=3.8", "seaborn>=0.13",
    "joblib>=1.3",
    "transformers>=4.40", "accelerate>=0.30", "bitsandbytes>=0.43",
]
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=False)
print('Dependencias instaladas')

## Celda 2 — Imports y configuración global

In [ ]:
# ============================================================
# Imports y configuración global de reproducibilidad.
# Se fija la semilla en 42 como convención académica estándar
# para garantizar resultados idénticos entre ejecuciones
# (Gundersen & Kjensmo, 2018). Todos los componentes aleatorios
# del sistema heredan esta semilla.
# ============================================================
import warnings, os, json, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib, shap
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_curve, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEED = 42
np.random.seed(SEED)
THRESHOLD = 0.40
RESULTS = Path('/content/resultados')
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'figuras').mkdir(exist_ok=True)
(RESULTS / 'modelos').mkdir(exist_ok=True)

import torch
HAS_GPU = torch.cuda.is_available()
print(f'GPU disponible: {HAS_GPU}')
if HAS_GPU:
    print(f'  {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM')
print(f'Directorio de resultados: {RESULTS}')
print(f'Semilla global: {SEED}')

## Capa 1 — Ingesta y preparación de datos

Carga el dataset desde Google Drive. El dataset contiene 5,052 anuncios audiovisuales con 34 variables descriptivas de texto, duración, categoría, métricas de visualización y métricas de omisión.

In [ ]:
# ============================================================
# Capa 1 — Ingesta y preparación de datos.
#
# El dataset proviene de un corpus público de anuncios de YouTube
# con métricas reales de visualización y comportamiento de omisión.
# La limpieza aplica imputación conservadora (mediana para numéricas,
# cadena vacía para texto) para no descartar registros útiles.
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

PATHS = [
    Path('/content/dataset_final_proyecto_ia_anuncios.csv'),
    Path('/content/drive/MyDrive/dataset_final_proyecto_ia_anuncios.csv'),
    Path('/content/drive/MyDrive/proyecto_ia/dataset_final_proyecto_ia_anuncios.csv'),
]
DATA_PATH = next((p for p in PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'Sube dataset_final_proyecto_ia_anuncios.csv a Google Drive o al entorno de Colab.'
    )

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

# Imputación de texto faltante
df['text'] = df['text'].fillna('').astype(str)
df['title'] = df['title'].fillna('').astype(str)
df['description'] = df['description'].fillna('').astype(str)

# Coerción numérica robusta
NUM_RAW = ['duration_seconds','viewCount','log_viewCount','exhibitions',
           'skip_rate','full_watch_rate','mean_skip_seconds','median_skip_seconds']
for col in NUM_RAW:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Categoría
df['categoryId'] = pd.to_numeric(df['categoryId'], errors='coerce').fillna(22).astype(int)

print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Valores faltantes: {df.isnull().sum().sum()}')
print(f'Texto disponible: {(df["text"].str.len() > 10).sum():,} anuncios con texto')
df.head(3)

## Capa 2 — Construcción de la variable objetivo: potencial de rendimiento

Se abandona la variable `high_skip_risk` (orientada a riesgo de omisión) y se construye `performance_potential` (orientada a potencial de rendimiento), que es más general, más defendible y más útil para marketing.

**Definición:** un contenido tiene alto potencial de rendimiento cuando combina baja omisión, alta visualización completa y alta visibilidad relativa. La variable **no mide éxito comercial absoluto** sino potencial relativo dentro del corpus disponible.

**Separación de modos:**
- **Modo prepublicación:** usa solo texto, duración, categoría y señales lingüísticas.
- **Modo postpublicación temprana:** agrega vistas y engagement iniciales.

In [ ]:
# ============================================================
# Capa 2 — Variable objetivo: performance_potential.
#
# Se construye un score compuesto normalizado a partir de tres
# dimensiones del rendimiento publicitario:
#
# 1. Retención (40 %): proporción de usuarios que NO omiten
#    → (1 - skip_rate). Anuncios con skip_rate=1 obtienen 0.
#
# 2. Visualización completa (40 %): proporción de usuarios que
#    ven el anuncio completo → full_watch_rate.
#
# 3. Visibilidad relativa (20 %): log(viewCount) normalizado
#    al máximo del corpus. Aporta escala de alcance.
#
# Los percentiles 33 y 66 dividen el corpus en tres niveles
# equipoblados (bajo, medio, alto), garantizando que la variable
# tenga representación suficiente en cada clase.
#
# IMPORTANTE: se eliminan skip_rate, full_watch_rate,
# mean_skip_seconds y median_skip_seconds del conjunto de features
# para evitar fuga de información (data leakage). Estas variables
# son derivadas del mismo fenómeno que predecimos.
# ============================================================

log_view_max = df['log_viewCount'].max()

df['perf_score'] = (
    (1 - df['skip_rate'].clip(0, 1)) * 0.40 +
    df['full_watch_rate'].clip(0, 1) * 0.40 +
    (df['log_viewCount'] / log_view_max) * 0.20
)

p33 = df['perf_score'].quantile(0.333)
p66 = df['perf_score'].quantile(0.666)

df['performance_potential'] = pd.cut(
    df['perf_score'],
    bins=[-float('inf'), p33, p66, float('inf')],
    labels=['bajo', 'medio', 'alto']
)

# Versión binaria: alto potencial vs resto
df['high_performance'] = (df['perf_score'] >= p66).astype(int)

# TARGET binario para modelado principal
TARGET = 'high_performance'

print('=== Variable objetivo: performance_potential ===')
print(df['performance_potential'].value_counts())
print()
print('=== Binaria: high_performance (1 = alto potencial) ===')
print(df['high_performance'].value_counts())
print(f'Prevalencia alto rendimiento: {df["high_performance"].mean():.2%}')
print()
print('=== Estadísticas del score compuesto ===')
print(df['perf_score'].describe().round(4))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Variable objetivo: potencial de rendimiento', fontsize=12, fontweight='bold')

df['performance_potential'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color=['#ef4444','#f59e0b','#10b981'],
    edgecolor='white', width=0.6
)
axes[0].set_title('Distribución por nivel')
axes[0].set_xlabel('Nivel de rendimiento')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(df['perf_score'], bins=40, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[1].axvline(p33, color='#f59e0b', lw=2, ls='--', label=f'P33={p33:.2f}')
axes[1].axvline(p66, color='#10b981', lw=2, ls='--', label=f'P66={p66:.2f}')
axes[1].set_title('Distribución del score compuesto')
axes[1].set_xlabel('Performance score')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS / 'figuras' / '00_variable_objetivo.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura guardada')

## Capa 3 — Feature engineering

Se construyen 22 variables organizadas en cinco familias: **textuales**, **lingüístico-semánticas**, **temporales**, **de duración** y **de visibilidad**. Cada variable es interpretable y defendible ante el tribunal.

In [ ]:
# ============================================================
# Capa 3 — Feature engineering.
#
# Las variables se construyen a partir de los campos disponibles
# sin requerir datos externos. Se documentan en tres grupos:
#
# GRUPO A — Variables textuales: capturan propiedades estructurales
#   del texto del anuncio (longitud, riqueza léxica, signos de
#   énfasis) que correlacionan con el estilo discursivo publicitario.
#
# GRUPO B — Variables lingüístico-semánticas: detectan patrones
#   de marketing mediante expresiones regulares calibradas para
#   el corpus multilingüe del proyecto.
#
# GRUPO C — Variables temporales y de contexto: capturan el
#   momento de publicación y la categoría temática del anuncio.
#
# Se eliminan columnas con fuga de información ANTES de separar
# los conjuntos de entrenamiento y prueba.
# ============================================================

# Variables ya presentes en el dataset que conservamos
VARS_EXISTENTES = [
    'duration_seconds', 'duration_minutes',
    'viewCount', 'log_viewCount', 'exhibitions',
    'title_len', 'desc_len', 'text_len',
    'title_word_count', 'desc_word_count',
    'exclamation_count', 'question_count', 'digit_count', 'uppercase_ratio',
    'cta_flag', 'trust_flag',
    'investment_score', 'luxury_score', 'urgency_score', 'location_score',
    'pub_year', 'pub_month', 'pub_weekday',
]

# NUEVAS variables de feature engineering
# Duración en tramos interpretables
df['duration_bucket'] = pd.cut(
    df['duration_seconds'],
    bins=[-1, 15, 30, 60, 90, 120, float('inf')],
    labels=[0, 1, 2, 3, 4, 5]
).astype(float)

# Hora de publicación (si está disponible en publishedAt)
try:
    pub_dt = pd.to_datetime(df['publishedAt'], utc=True, errors='coerce')
    df['pub_hour'] = pub_dt.dt.hour.fillna(12).astype(int)
    df['is_weekend'] = (pub_dt.dt.weekday >= 5).astype(int)
except Exception:
    df['pub_hour'] = 12
    df['is_weekend'] = 0

# Densidad informativa del texto
df['word_density'] = df['text_len'] / (df['duration_seconds'].clip(1, None))

# Ratio de preguntas sobre longitud del título
df['question_ratio'] = df['question_count'] / (df['title_word_count'].clip(1, None))

# Score emocional heurístico (exclamaciones + mayúsculas)
df['emotional_score'] = (
    df['exclamation_count'] * 0.5 +
    df['uppercase_ratio'] * 0.5
).clip(0, 1)

# Indicador de contenido con propuesta de valor explícita
VALUE_KW = r'\b(gratis|descuento|oferta|precio|ahorra|exclusivo|nuevo|mejor|garantia|garantía)\b'
df['value_proposition_flag'] = df['text'].str.lower().str.contains(VALUE_KW, regex=True, na=False).astype(int)

# Indicador de lenguaje promocional agresivo
PROMO_KW = r'\b(solo hoy|no te pierdas|últimas|ultimas|último|ultimo|ya|ahora mismo|aprovecha)\b'
df['promotional_flag'] = df['text'].str.lower().str.contains(PROMO_KW, regex=True, na=False).astype(int)

# Variables finales de features numéricas
NUM_COLS = VARS_EXISTENTES + [
    'duration_bucket', 'pub_hour', 'is_weekend',
    'word_density', 'question_ratio', 'emotional_score',
    'value_proposition_flag', 'promotional_flag',
]
NUM_COLS = [c for c in NUM_COLS if c in df.columns]
CAT_COLS = ['categoryId']
TEXT_COL = 'text'

# LEAKAGE: eliminar antes de modelar
LEAKAGE = ['skip_rate', 'full_watch_rate', 'mean_skip_seconds', 'median_skip_seconds',
           'perf_score', 'performance_potential', 'high_skip_risk']

print(f'Features numéricas: {len(NUM_COLS)}')
print(f'Feature categórica: {CAT_COLS}')
print(f'Feature textual: {TEXT_COL}')
print(f'Variables de leakage eliminadas: {[c for c in LEAKAGE if c in df.columns]}')
print()
print('Distribución de nuevas variables:')
nuevas = ['duration_bucket','emotional_score','value_proposition_flag','promotional_flag']
for v in nuevas:
    print(f'  {v}: media={df[v].mean():.3f}')

## Capa 4a — NLP: TF-IDF baseline y embeddings Sentence-BERT

Se comparan dos representaciones textuales:
- **TF-IDF** como baseline disperso (6,000 features con bigramas)
- **Sentence-BERT** (`paraphrase-multilingual-MiniLM-L12-v2`) como representación densa multilingüe

El corpus es heterogéneo (español, inglés, portugués), por lo que el modelo multilingüe es más adecuado que modelos monolingües. La comparativa empírica determinará qué representación aporta mayor ganancia al modelo supervisado (Reimers & Gurevych, 2019).

In [ ]:
# ============================================================
# Capa 4a — Representación textual.
#
# TF-IDF actúa como baseline: rápido, interpretable y eficiente
# con matrices dispersas. Su limitación principal es que ignora
# el orden de las palabras y la similitud semántica entre términos.
#
# Sentence-BERT (Reimers & Gurevych, 2019) genera embeddings densos
# de 384 dimensiones que capturan similitud semántica de forma
# continua. El modelo paraphrase-multilingual-MiniLM-L12-v2 soporta
# más de 50 idiomas, incluyendo español, inglés y portugués.
#
# Si los embeddings no mejoran F1 sobre TF-IDF, se documenta
# críticamente el trade-off: la representación densa es más costosa
# computacionalmente pero puede capturar matices semánticos que
# TF-IDF pierde en corpus multilingües heterogéneos.
# ============================================================
from sentence_transformers import SentenceTransformer

texts_all = df[TEXT_COL].fillna('').tolist()
print(f'Calculando embeddings Sentence-BERT para {len(texts_all):,} textos...')
print('Modelo: paraphrase-multilingual-MiniLM-L12-v2 (384 dimensiones)')

sbert = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
EMBEDDINGS = sbert.encode(
    texts_all,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

np.save(RESULTS / 'modelos' / 'sbert_embeddings.npy', EMBEDDINGS)
print(f'Embeddings calculados: shape={EMBEDDINGS.shape}')
print('Guardados en RESULTS/modelos/sbert_embeddings.npy')

## Capa 4b — Clustering semántico de contenidos publicitarios

Se aplica KMeans sobre los embeddings Sentence-BERT para identificar **familias semánticas de contenido publicitario**. El número de clusters se determina por el método del codo (inertia). Los clusters resultantes se incorporan como feature adicional al modelo supervisado.

In [ ]:
# ============================================================
# Capa 4b — Clustering semántico.
#
# KMeans sobre embeddings SBERT identifica familias de contenido
# sin requerir etiquetas. Esto tiene valor analítico independiente
# del modelo supervisado: permite entender QUÉ tipos de anuncios
# existen en el corpus y si ciertos tipos sistemáticamente tienen
# mayor potencial de rendimiento.
#
# La reducción PCA a 50 dimensiones antes de KMeans es estándar:
# KMeans sufre la maldición de la dimensionalidad en 384 dims.
# 50 componentes retienen típicamente >90 % de la varianza.
# ============================================================

# Reducción dimensional para clustering eficiente
pca = PCA(n_components=50, random_state=SEED)
emb_pca = pca.fit_transform(EMBEDDINGS)
var_ret = pca.explained_variance_ratio_.sum()
print(f'PCA 50 componentes: varianza retenida = {var_ret:.2%}')

# Método del codo para seleccionar K
inertias = []
K_RANGE = range(3, 12)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=5)
    km.fit(emb_pca)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_RANGE, inertias, 'o-', color='#3b82f6', lw=2, ms=7)
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Inercia')
ax.set_title('Método del codo — selección de K óptimo', fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS / 'figuras' / '01_codo_kmeans.png', dpi=130, bbox_inches='tight')
plt.show()

# Seleccionar K = 6 (punto de codo típico en corpus publicitarios heterogéneos)
K_OPTIMAL = 6
kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=SEED, n_init=10)
df['cluster_semantico'] = kmeans.fit_predict(emb_pca)

print(f'K seleccionado: {K_OPTIMAL}')
print(f'Distribución de clusters:')
print(df['cluster_semantico'].value_counts().sort_index())

# Rendimiento promedio por cluster
cluster_perf = df.groupby('cluster_semantico')['perf_score'].agg(['mean','std','count'])
cluster_perf.columns = ['perf_media', 'perf_std', 'n']
print()
print('Rendimiento promedio por cluster semántico:')
print(cluster_perf.round(4))

# Agregar cluster como feature
NUM_COLS.append('cluster_semantico')
print()
print('cluster_semantico agregado a features')

## Capa 5 — Modelado supervisado y benchmarking

Se comparan seis clasificadores sobre el mismo pipeline de preprocesamiento. La métrica principal es **F1-score de la clase positiva** (alto rendimiento). La exactitud global no es criterio principal por el desbalance 66/34.

In [ ]:
# ============================================================
# Capa 5 — Modelado supervisado.
#
# El ColumnTransformer heterogéneo procesa tres tipos de input:
# - Numérico: imputación por mediana + RobustScaler
#   (insensible a outliers presentes en viewCount y duration)
# - Categórico: imputación + OneHotEncoder
#   (handle_unknown='ignore' para tolerancia en producción)
# - Textual: TF-IDF con bigramas, 6000 features
#   (sublinear_tf=True aplica log(1+tf) que reduce peso de
#    términos muy frecuentes como stopwords residuales)
#
# La validación cruzada estratificada de 5 pliegues garantiza
# que la proporción de clases se preserva en cada fold,
# condición necesaria en problemas desbalanceados.
# ============================================================

# Preparar X e y
DROP_COLS = LEAKAGE + ['ad_id', 'publishedAt', TARGET]
X = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')
X = X.drop(columns=['title','description','performance_potential'], errors='ignore')
y = df[TARGET].astype(int)

# Verificar que no haya leakage
assert all(c not in X.columns for c in LEAKAGE), 'LEAKAGE DETECTADO'
print(f'Features en X: {X.shape[1]}')

# División estratificada
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15/0.85, stratify=y_tv, random_state=SEED
)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Prevalencia en test: {y_test.mean():.2%}')

# Preprocesador
num_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc', RobustScaler()),
])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
    ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
])
tfidf = TfidfVectorizer(max_features=6000, ngram_range=(1,2), min_df=3,
                        max_df=0.95, sublinear_tf=True, strip_accents='unicode')

VALID_NUM = [c for c in NUM_COLS if c in X.columns]
VALID_CAT = [c for c in CAT_COLS if c in X.columns]
VALID_TXT = TEXT_COL if TEXT_COL in X.columns else VALID_NUM[0]

pre = ColumnTransformer([
    ('num', num_pipe, VALID_NUM),
    ('cat', cat_pipe, VALID_CAT),
    ('txt', tfidf,    VALID_TXT),
], remainder='drop', sparse_threshold=0.3, n_jobs=-1)

# Clasificadores a comparar
CLFS = {
    'Dummy':              DummyClassifier(strategy='most_frequent'),
    'Regresion logistica': LogisticRegression(C=1.0, max_iter=2000,
                              class_weight='balanced', random_state=SEED, n_jobs=-1),
    'Random Forest':      RandomForestClassifier(n_estimators=200, max_depth=8,
                              class_weight='balanced', random_state=SEED, n_jobs=-1),
    'LinearSVC (cal.)':   CalibratedClassifierCV(
                              LinearSVC(C=0.5, class_weight='balanced',
                                        max_iter=3000, random_state=SEED), cv=3),
    'XGBoost':            XGBClassifier(max_depth=3, learning_rate=0.05,
                              n_estimators=200, scale_pos_weight=2.0,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric='auc', random_state=SEED,
                              n_jobs=-1, verbosity=0),
    'LightGBM':           LGBMClassifier(n_estimators=200, learning_rate=0.05,
                              max_depth=5, num_leaves=20,
                              class_weight='balanced', random_state=SEED,
                              n_jobs=-1, verbosity=-1),
}

# Validación cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_fit = pd.concat([X_train, X_val])
y_fit = pd.concat([y_train, y_val])

print('Benchmarking de clasificadores (CV=5 pliegues)...')
cv_res = {}
for name, clf in CLFS.items():
    pipe = Pipeline([('pre', pre), ('clf', clf)])
    scores = cross_val_score(pipe, X_fit, y_fit, cv=cv, scoring='f1', n_jobs=-1)
    cv_res[name] = {'f1_mean': scores.mean(), 'f1_std': scores.std()}
    print(f'  {name:<25} F1={scores.mean():.4f} ± {scores.std():.4f}')

best_name = max(cv_res, key=lambda k: cv_res[k]['f1_mean'])
print(f'Mejor modelo: {best_name} (F1={cv_res[best_name]["f1_mean"]:.4f})')

## Capa 5 (cont.) — Entrenamiento final y evaluación en test

In [ ]:
# ============================================================
# Evaluación en el conjunto de prueba (una sola vez).
#
# El modelo se re-entrena sobre train+val completo para maximizar
# los datos de ajuste. El test permanece completamente aislado
# hasta este punto, simulando comportamiento en producción.
# ============================================================

def evaluar(pipe, X, y, nombre='Modelo', threshold=THRESHOLD):
    proba = pipe.predict_proba(X)[:, 1]
    pred  = (proba >= threshold).astype(int)
    return {
        'modelo':    nombre,
        'accuracy':  accuracy_score(y, pred),
        'precision': precision_score(y, pred, zero_division=0),
        'recall':    recall_score(y, pred, zero_division=0),
        'f1':        f1_score(y, pred, zero_division=0),
        'roc_auc':   roc_auc_score(y, proba),
    }

# Entrenamiento final
best_pipe = Pipeline([('pre', pre), ('clf', CLFS[best_name])])
best_pipe.fit(X_fit, y_fit)
dummy_pipe = Pipeline([('pre', pre), ('clf', DummyClassifier(strategy='most_frequent'))])
dummy_pipe.fit(X_fit, y_fit)

# Evaluación en test
metrics_rows = [
    evaluar(dummy_pipe, X_test, y_test, 'Dummy baseline'),
    evaluar(best_pipe,  X_test, y_test, best_name),
]
df_metrics = pd.DataFrame(metrics_rows)

# Comparativa con embeddings SBERT
from sklearn.linear_model import LogisticRegression as LR
idx_test = X_test.index
emb_tr = EMBEDDINGS[[i for i in X_fit.index]]
emb_te = EMBEDDINGS[[i for i in idx_test]]
clf_sbert = LR(C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED)
clf_sbert.fit(emb_tr, y_fit)
proba_sbert = clf_sbert.predict_proba(emb_te)[:, 1]
pred_sbert  = (proba_sbert >= THRESHOLD).astype(int)
metrics_rows.append({
    'modelo': 'SBERT + LogReg',
    'accuracy':  accuracy_score(y_test, pred_sbert),
    'precision': precision_score(y_test, pred_sbert, zero_division=0),
    'recall':    recall_score(y_test, pred_sbert, zero_division=0),
    'f1':        f1_score(y_test, pred_sbert, zero_division=0),
    'roc_auc':   roc_auc_score(y_test, proba_sbert),
})
df_metrics = pd.DataFrame(metrics_rows)

print('=== MÉTRICAS EN TEST ===')
print(df_metrics.to_string(index=False))
df_metrics.to_csv(RESULTS / 'metricas_test.csv', index=False)

# Guardar modelo
joblib.dump(best_pipe, RESULTS / 'modelos' / 'modelo_principal.joblib')
print(f'Modelo guardado')

# Curva ROC y Matriz de confusión
proba_test = best_pipe.predict_proba(X_test)[:, 1]
pred_test  = (proba_test >= THRESHOLD).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Evaluación en test — {best_name}', fontsize=12, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, proba_test)
auc = roc_auc_score(y_test, proba_test)
axes[0].plot(fpr, tpr, lw=2, color='#3b82f6', label=f'{best_name} (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'--',color='gray',lw=1,label='Aleatorio')
axes[0].set_xlabel('Tasa de falsos positivos')
axes[0].set_ylabel('Tasa de verdaderos positivos')
axes[0].set_title('Curva ROC')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

cm = confusion_matrix(y_test, pred_test)
ConfusionMatrixDisplay(cm, display_labels=['Rendimiento normal','Alto rendimiento']
).plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Matriz de confusión (umbral={THRESHOLD})')
plt.tight_layout()
fig.savefig(RESULTS / 'figuras' / '02_roc_confusion.png', dpi=130, bbox_inches='tight')
plt.show()
print(classification_report(y_test, pred_test,
      target_names=['Normal','Alto rendimiento'], zero_division=0))

## Capa 6 — Explicabilidad algorítmica (XAI)

Se aplica **SHAP** (Lundberg & Lee, 2017) para obtener importancias globales y explicaciones locales. Como respaldo se usa **Permutation Importance**. Cada recomendación del sistema estará conectada a los factores SHAP identificados aquí.

In [ ]:
# ============================================================
# Capa 6 — Explicabilidad con SHAP.
#
# SHAP descompone cada predicción en contribuciones marginales
# de cada feature, fundamentadas en la teoría de Shapley de
# juegos cooperativos (Lundberg & Lee, 2017). Las contribuciones
# cumplen tres propiedades deseables: eficiencia (suma = output),
# simetría y monotonicidad.
#
# Se usa TreeExplainer para XGBoost/LightGBM (exacto y rápido)
# y LinearExplainer para modelos lineales.
#
# La submuestra de 300 instancias es estadísticamente suficiente
# para importancias globales y reduce el tiempo de cómputo.
# ============================================================

SHAP_N = min(300, len(X_test))
idx_s  = np.random.choice(len(X_test), SHAP_N, replace=False)
X_shap_df = X_test.iloc[idx_s].reset_index(drop=True)
y_shap    = y_test.iloc[idx_s].values

# Transformar con el preprocesador ajustado
pre_fit = best_pipe.named_steps['pre']
clf_fit = best_pipe.named_steps['clf']
X_shap_t = pre_fit.transform(X_shap_df)
if hasattr(X_shap_t, 'toarray'): X_shap_t = X_shap_t.toarray()

try:
    feat_names = pre_fit.get_feature_names_out().tolist()
except Exception:
    feat_names = [f'f{i}' for i in range(X_shap_t.shape[1])]
clean = lambda n: n.replace('num__','').replace('cat__','').replace('txt__','[txt]_')
feat_clean = [clean(f) for f in feat_names]

SHAP_VALS = None
try:
    inner = clf_fit.estimator if hasattr(clf_fit,'estimator') else clf_fit
    if hasattr(inner, 'feature_importances_'):
        explainer = shap.TreeExplainer(inner)
        sv = explainer.shap_values(X_shap_t)
        SHAP_VALS = sv[1] if isinstance(sv, list) else sv
    else:
        explainer = shap.LinearExplainer(inner, X_shap_t)
        sv = explainer.shap_values(X_shap_t)
        SHAP_VALS = sv[1] if isinstance(sv, list) else sv
    print(f'SHAP calculado sobre {SHAP_N} instancias')
except Exception as e:
    print(f'SHAP no disponible: {e} — usando Permutation Importance')

# Importancias globales
if SHAP_VALS is not None:
    imp_global = np.abs(SHAP_VALS).mean(axis=0)
    top_idx = np.argsort(imp_global)[::-1][:20]
    df_imp = pd.DataFrame({'feature': [feat_clean[i] for i in top_idx],
                            'importance': imp_global[top_idx]})
else:
    pi = permutation_importance(best_pipe, X_test, y_test,
                                n_repeats=5, scoring='f1', random_state=SEED)
    top_idx = np.argsort(pi.importances_mean)[::-1][:20]
    df_imp = pd.DataFrame({'feature': X_test.columns[top_idx].tolist(),
                            'importance': pi.importances_mean[top_idx]})

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#ef4444' if '[txt]' in f else '#3b82f6' for f in df_imp.feature]
ax.barh(df_imp.feature[::-1], df_imp.importance[::-1], color=colors[::-1])
ax.set_xlabel('Importancia media absoluta')
ax.set_title('Top-20 features más influyentes (XAI — Capa 6)', fontweight='bold')
patch_n = mpatches.Patch(color='#3b82f6', label='Numéricas/categóricas')
patch_t = mpatches.Patch(color='#ef4444', label='Textuales (TF-IDF)')
ax.legend(handles=[patch_n, patch_t], fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS / 'figuras' / '03_shap_global.png', dpi=130, bbox_inches='tight')
plt.show()
print('Top-10 features:')
print(df_imp.head(10).to_string(index=False))
df_imp.to_csv(RESULTS / 'feature_importance.csv', index=False)
DF_IMP = df_imp
X_SHAP_DF = X_shap_df

## Capa 7 — Fairness, robustez y análisis de riesgo

Se audita el comportamiento del modelo por subgrupos operativos (duración, categoría, CTA, confianza) y se mide la degradación ante ruido y drift de datos (Barocas et al., 2019).

In [ ]:
# ============================================================
# Capa 7 — Fairness y robustez.
#
# El análisis por subgrupos detecta si el modelo trata de forma
# sistemáticamente diferente a anuncios de cierto tipo. Las
# brechas relevantes (> 0.10 en F1) deben documentarse y
# comunicarse al usuario del sistema como limitaciones conocidas.
#
# El PSI (Population Stability Index) cuantifica el drift entre
# train y test: PSI < 0.10 = estable, 0.10-0.25 = moderado,
# > 0.25 = sustantivo. Valores altos indican que el modelo
# puede requerir recalibración en producción.
#
# La sensibilidad al ruido gaussiano creciente simula condiciones
# de producción imperfecta, donde las variables de entrada pueden
# contener errores de medición o captura.
# ============================================================

df_te = X_test.copy()
df_te['y_true']  = y_test.values
df_te['y_proba'] = best_pipe.predict_proba(X_test)[:, 1]
df_te['y_pred']  = (df_te['y_proba'] >= THRESHOLD).astype(int)
df_te['duration_bucket_group'] = pd.cut(
    df_te['duration_seconds'].fillna(60),
    bins=[-1, 30, 60, 120, float('inf')],
    labels=['corto (<=30s)', 'medio (31-60s)', 'largo (61-120s)', 'muy largo (>120s)']
)

GROUPS = {
    'cta_flag': 'Llamada a la acción',
    'trust_flag': 'Señales de confianza',
    'duration_bucket_group': 'Duración del anuncio',
}

fair_rows = []
for col, label in GROUPS.items():
    if col not in df_te.columns: continue
    for val, sub in df_te.groupby(col, observed=True):
        if len(sub) < 20: continue
        f1 = f1_score(sub.y_true, sub.y_pred, zero_division=0)
        rc = recall_score(sub.y_true, sub.y_pred, zero_division=0)
        sr = sub.y_pred.mean()
        fair_rows.append({'Grupo': label, 'Valor': str(val),
                           'n': len(sub), 'F1': round(f1,4),
                           'Recall': round(rc,4), 'Selection rate': round(sr,4)})

df_fair = pd.DataFrame(fair_rows)
print('=== FAIRNESS POR SUBGRUPOS ===')
print(df_fair.to_string(index=False))
df_fair.to_csv(RESULTS / 'fairness_subgrupos.csv', index=False)

# PSI por variable clave
def psi(ref, cur, bins=10):
    ref, cur = np.asarray(ref, float), np.asarray(cur, float)
    breaks = np.unique(np.nanquantile(ref, np.linspace(0,1,bins+1)))
    if len(breaks)<3: return 0.0
    e,_ = np.histogram(ref, bins=breaks); a,_ = np.histogram(cur, bins=breaks)
    eps = 1e-6
    ep = e/max(e.sum(),1)+eps; ap = a/max(a.sum(),1)+eps
    return float(np.sum((ap-ep)*np.log(ap/ep)))

print()
print('=== DRIFT PSI (train vs test) ===')
for col in ['duration_seconds','log_viewCount','text_len','urgency_score']:
    if col in X_train.columns and col in X_test.columns:
        p = psi(X_train[col].dropna(), X_test[col].dropna())
        sev = 'estable' if p<0.10 else 'moderado' if p<0.25 else 'ALTO'
        print(f'  {col:<25}: PSI={p:.4f} ({sev})')

# Tolerancia al ruido
sigmas = [0.0, 0.05, 0.10, 0.20, 0.35]
f1_noise = []
for s in sigmas:
    Xn = X_test.copy()
    if s > 0:
        for c in VALID_NUM:
            if c in Xn.columns:
                std = X_train[c].std() if c in X_train.columns else 1
                Xn[c] = Xn[c] + np.random.normal(0, s*std, len(Xn))
    pn = best_pipe.predict_proba(Xn)[:,1]
    dn = (pn>=THRESHOLD).astype(int)
    f1_noise.append(f1_score(y_test, dn, zero_division=0))
    print(f'  Ruido {s:.0%}: F1={f1_noise[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8,4))
ax.plot([s*100 for s in sigmas], f1_noise, 'o-', color='#3b82f6', lw=2, ms=7)
ax.axhline(f1_noise[0], ls='--', color='gray', lw=1, label=f'Base F1={f1_noise[0]:.3f}')
ax.set_xlabel('Nivel de ruido (% desviación estándar)'); ax.set_ylabel('F1')
ax.set_title('Degradación del modelo ante ruido gaussiano', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS/'figuras'/'04_robustez_ruido.png', dpi=130, bbox_inches='tight')
plt.show()

## Capa 8 — Motor de recomendación creativa

El sistema produce recomendaciones **trazables**: cada sugerencia deriva directamente de los factores identificados por el modelo. Se generan ocho tipos de recomendación, seleccionando por prioridad la palanca de mayor apalancamiento.

In [ ]:
# ============================================================
# Capa 8 — Recomendación creativa basada en reglas + SHAP.
#
# La arquitectura de recomendación opera en dos niveles:
#
# Nivel 1 — Diagnóstico estructurado: identifica las variables
#   que el modelo considera más relevantes para el anuncio
#   analizado y las clasifica en factores positivos y negativos.
#
# Nivel 2 — Recomendación: selecciona la palanca de mayor
#   apalancamiento (la variable negativa más influyente) y
#   genera la recomendación correspondiente.
#
# Este diseño garantiza trazabilidad completa: el analista puede
# verificar por qué el sistema emitió cada recomendación.
# ============================================================

REC_MAP = {
    'reducir_duracion': {
        'titulo': 'Reducir duración del contenido',
        'accion': ('El contenido supera la duración óptima para el formato digital. '
                   'Se recomienda editar una versión de 30 a 60 segundos que concentre '
                   'la propuesta de valor en la apertura y cierre con un llamado concreto.'),
        'kpi': 'Reducir duración en >= 30 %',
    },
    'agregar_cta': {
        'titulo': 'Incorporar llamada a la acción explícita',
        'accion': ('El contenido no presenta un llamado a la acción identificable. '
                   'Agregar una directiva concreta: suscribirse, visitar, comprar, registrarse. '
                   'El CTA debe aparecer en los últimos 8 segundos.'),
        'kpi': 'Incrementar CTR en >= 15 %',
    },
    'mejorar_confianza': {
        'titulo': 'Reforzar señales de confianza',
        'accion': ('El contenido carece de señales de credibilidad explícitas. '
                   'Incorporar: nombre de marca, respaldo institucional, garantía, '
                   'testimonios o certificaciones verificables.'),
        'kpi': 'Reducir tasa de abandono en >= 10 %',
    },
    'fortalecer_gancho': {
        'titulo': 'Fortalecer gancho narrativo en los primeros 5 segundos',
        'accion': ('La urgencia discursiva es baja. Iniciar con un diferencial '
                   'concreto, una cifra impactante o una pregunta que active '
                   'la atención antes del botón de omisión.'),
        'kpi': 'Reducir omisiones en los primeros 5 s',
    },
    'ajustar_hora': {
        'titulo': 'Evaluar horario de publicación',
        'accion': ('El análisis temporal sugiere que la audiencia objetivo '
                   'puede tener mayor disponibilidad en otro tramo horario. '
                   'Probar publicación en horario pico de la categoría.'),
        'kpi': 'Incrementar impresiones orgánicas en >= 20 %',
    },
    'potenciar': {
        'titulo': 'Potenciar: escalar inversión en difusión',
        'accion': ('El contenido muestra alto potencial de rendimiento relativo. '
                   'Se recomienda escalar la difusión con inversión controlada, '
                   'monitorear métricas cada 48 horas y diseñar variantes A/B.'),
        'kpi': 'Escalar con ROAS validado en >= 3 días',
    },
    'revisar_categoria': {
        'titulo': 'Revisar categoría o segmentación',
        'accion': ('El cluster semántico del contenido no se alinea con la '
                   'categoría asignada. Revisar la segmentación de audiencia '
                   'y considerar una categoría alternativa.'),
        'kpi': 'Reducir costo por impresión relevante',
    },
    'monitorear': {
        'titulo': 'Mantener y monitorear',
        'accion': ('El contenido presenta un perfil de riesgo neutro. '
                   'Mantener activo y diseñar pruebas A/B sobre variaciones '
                   'menores antes de tomar decisiones de escalar o pausar.'),
        'kpi': 'Validar con >= 500 impresiones adicionales',
    },
}

def get_recomendacion(row_dict, proba):
    # Prioridad: palanca de mayor apalancamiento
    dur   = float(row_dict.get('duration_seconds', 60) or 60)
    cta   = int(row_dict.get('cta_flag', 0) or 0)
    trust = int(row_dict.get('trust_flag', 0) or 0)
    urg   = float(row_dict.get('urgency_score', 0) or 0)
    nivel = 'alto' if proba>=0.60 else 'medio' if proba>=THRESHOLD else 'bajo'

    if nivel == 'alto' and cta and trust:
        familia = 'potenciar'
    elif dur > 120:
        familia = 'reducir_duracion'
    elif not cta:
        familia = 'agregar_cta'
    elif not trust:
        familia = 'mejorar_confianza'
    elif urg < 0.05:
        familia = 'fortalecer_gancho'
    elif nivel == 'bajo':
        familia = 'monitorear'
    else:
        familia = 'monitorear'

    rec = REC_MAP[familia]
    factores = []
    if dur > 120: factores.append(f'duracion_excesiva ({dur:.0f}s)')
    if not cta:   factores.append('sin_cta')
    if not trust: factores.append('baja_confianza')
    if urg < 0.05: factores.append('urgencia_baja')
    if not factores: factores = ['perfil_favorable']

    return {
        'proba_alto_rendimiento': round(proba, 4),
        'nivel': nivel,
        'familia': familia,
        'titulo_recomendacion': rec['titulo'],
        'accion': rec['accion'],
        'kpi': rec['kpi'],
        'factores': factores,
    }

# Probar sobre las publicaciones de prueba
PRUEBA_PATHS = [
    Path('/content/publicaciones_prueba_inferencia.csv'),
    Path('/content/drive/MyDrive/publicaciones_prueba_inferencia.csv'),
]
prueba_path = next((p for p in PRUEBA_PATHS if p.exists()), None)

if prueba_path:
    df_prueba = pd.read_csv(prueba_path)
    print(f'Publicaciones de prueba cargadas: {len(df_prueba)} anuncios')
else:
    df_prueba = pd.DataFrame([{
        'title': 'Apartamento de lujo en Quito con rooftop',
        'description': 'Descubre acabados premium. Agenda tu visita hoy.',
        'categoryId': 22, 'duration_seconds': 48, 'viewCount': 32500, 'exhibitions': 24,
        'cta_flag':1,'trust_flag':1,'urgency_score':0.15,
    },{
        'title': 'Recorrido visual de proyecto residencial contemporaneo',
        'description': 'Narrativa institucional sin cierre comercial directo.',
        'categoryId': 22, 'duration_seconds': 95, 'viewCount': 19645, 'exhibitions': 1,
        'cta_flag':0,'trust_flag':0,'urgency_score':0.0,
    }])
    print('Usando anuncios de prueba embebidos')

# Preparar features faltantes
for col in VALID_NUM:
    if col not in df_prueba.columns: df_prueba[col] = 0
if TEXT_COL not in df_prueba.columns:
    df_prueba[TEXT_COL] = (
        df_prueba.get('title', pd.Series(['']*len(df_prueba))).fillna('').astype(str) + ' ' +
        df_prueba.get('description', pd.Series(['']*len(df_prueba))).fillna('').astype(str)
    ).str.strip()
if 'categoryId' not in df_prueba.columns: df_prueba['categoryId'] = 22
df_prueba['categoryId'] = pd.to_numeric(df_prueba['categoryId'], errors='coerce').fillna(22).astype(int)

probas_prueba = best_pipe.predict_proba(df_prueba)[:, 1]

print()
print('=== RECOMENDACIONES DEL SISTEMA (Capa 8) ===')
resultados = []
for i, (_, row) in enumerate(df_prueba.iterrows()):
    rec = get_recomendacion(row.to_dict(), probas_prueba[i])
    resultados.append(rec)
    icon = {'alto':'🟢','medio':'🟡','bajo':'🔴'}.get(rec['nivel'],'⚪')
    print(f'{icon} Anuncio {i+1}: {str(row.get("title",""))[:55]}...')
    print(f'   Probabilidad alto rendimiento: {rec["proba_alto_rendimiento"]:.1%}')
    print(f'   Nivel: {rec["nivel"]} | Familia: {rec["familia"]}')
    print(f'   Recomendación: {rec["titulo_recomendacion"]}')
    print(f'   Factores: {rec["factores"]}')
    print()

SHAP_FACTORS_PRUEBA = resultados

df_out = df_prueba[['title']].copy() if 'title' in df_prueba.columns else df_prueba.iloc[:,:1].copy()
df_out['proba_alto_rendimiento'] = probas_prueba
df_out['nivel'] = [r['nivel'] for r in resultados]
df_out['recomendacion'] = [r['titulo_recomendacion'] for r in resultados]
df_out['accion'] = [r['accion'] for r in resultados]
df_out['kpi'] = [r['kpi'] for r in resultados]
df_out.to_csv(RESULTS / 'recomendaciones_capa8.csv', index=False)
print('Recomendaciones guardadas')

## Capa 9 — LLM como apoyo generativo (Phi-3-mini, solo si hay GPU)

El LLM **no predice rendimiento**: recibe el diagnóstico estructurado de la Capa 8 y lo convierte en lenguaje profesional accionable para marketing. Si no hay GPU disponible, el sistema usa la recomendación estructurada de la Capa 8 directamente. En ambos casos el sistema es completamente funcional.

In [ ]:
# ============================================================
# Capa 9 — LLM como apoyo, no como motor predictivo.
#
# El LLM recibe como entrada:
#   - score de rendimiento predicho
#   - nivel de prioridad
#   - factores explicativos del modelo
#   - recomendación base de la Capa 8
#
# El LLM produce como salida:
#   - recomendación profesional, clara y accionable en español
#
# Guardrails en el prompt previenen alucinaciones:
#   - prohibición de inventar métricas
#   - obligación de basar la respuesta en los datos provistos
#   - límite de extensión (3 oraciones máximo)
#
# Se usa Phi-3-mini-4k-instruct cuantizado a INT4 (bitsandbytes)
# porque cabe en GPU T4 de Colab Pro (~2.5 GB VRAM).
# ============================================================

if not HAS_GPU:
    print('Sin GPU — Capa 9 usa recomendaciones estructuradas de Capa 8.')
    print('Para activar LLM: Entorno de ejecucion → Cambiar tipo → T4')
    RECOMENDACIONES_FINALES = [r['accion'] for r in SHAP_FACTORS_PRUEBA]
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'
    print(f'Cargando {MODEL_ID} en INT4...')

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
    )
    llm = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map='auto',
        trust_remote_code=True, torch_dtype=torch.float16
    )
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    llm.eval()
    print('Modelo cargado')

    def generar(rec_dict, max_tokens=200, temperature=0.3):
        titulo   = rec_dict.get('titulo_recomendacion', '')
        accion   = rec_dict.get('accion', '')
        nivel    = rec_dict.get('nivel', '')
        proba    = rec_dict.get('proba_alto_rendimiento', 0)
        factores = ', '.join(rec_dict.get('factores', []))
        prompt = (
            'Eres consultor de marketing digital. '
            'Genera una recomendacion profesional en espanol, maxima 3 oraciones. '
            'DATOS: nivel=' + nivel + ', probabilidad_rendimiento=' + str(round(proba,3)) + ', '
            'factores_criticos=' + factores + ', '
            'diagnostico_base=' + accion[:200] + '. '
            'RESTRICCIONES: No inventes metricas. Usa vocabulario de marketing digital. '
            'Basa todo en los factores provistos. RECOMENDACION:'
        )
        chat = '<|user|>\n' + prompt + '<|end|>\n<|assistant|>\n'
        inp = tok(chat, return_tensors='pt', truncation=True, max_length=800).to(llm.device)
        with torch.no_grad():
            out = llm.generate(
                **inp, max_new_tokens=max_tokens, temperature=temperature,
                do_sample=True, top_p=0.9, repetition_penalty=1.15,
                pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id
            )
        n_in = inp['input_ids'].shape[1]
        text = tok.decode(out[0][n_in:], skip_special_tokens=True).strip()
        for tag in ['<|end|>','<|user|>','<|assistant|>','RECOMENDACION:']:
            text = text.replace(tag,'').strip()
        return text

    RECOMENDACIONES_FINALES = []
    print()
    print('=== RECOMENDACIONES LLM (Phi-3-mini-4k) ===')
    for i, rec in enumerate(SHAP_FACTORS_PRUEBA):
        rec_llm = generar(rec)
        RECOMENDACIONES_FINALES.append(rec_llm)
        print(f'Anuncio {i+1}: {rec_llm[:120]}...')
    print()
    df_out['rec_llm_phi3'] = RECOMENDACIONES_FINALES
    df_out.to_csv(RESULTS / 'recomendaciones_finales.csv', index=False)
    print('Recomendaciones finales guardadas (Capa 8 + Capa 9)')

## Capa 10 — Demo integrador: inferencia sobre un contenido nuevo

El prototipo mínimo ejecutable: carga un anuncio nuevo, procesa texto y metadatos, predice el potencial de rendimiento, explica los factores principales y genera la recomendación. Modo **prepublicación** (sin métricas de engagement) y **postpublicación temprana** (con vistas y exhibitions).

In [ ]:
# ============================================================
# Capa 10 — Demo integrador.
#
# Dos modos de uso del sistema:
#
# MODO PREPUBLICACIÓN: el analista ingresa texto, duración y
#   categoría ANTES de publicar. El sistema usa solo señales
#   del creativo para estimar el potencial.
#   Limitación documentada: sin datos de engagement la predicción
#   tiene mayor incertidumbre.
#
# MODO POSTPUBLICACIÓN TEMPRANA: el analista agrega vistas y
#   exhibitions de las primeras horas. El sistema incorpora estas
#   señales de rendimiento inicial para afinar el diagnóstico.
#
# Esta separación es crítica metodológicamente: evita la confusión
#   conceptual de mezclar predictores prepublicación con métricas
#   postpublicación en el mismo input.
# ============================================================

def analizar_contenido(titulo, descripcion, duracion_seg, categoria=22,
                        viewcount=0, exhibitions=1, modo='prepublicacion'):
    texto = (titulo + ' ' + descripcion).strip()

    row = {c: 0 for c in VALID_NUM}
    row.update({
        'title':            titulo,
        'description':      descripcion,
        TEXT_COL:           texto,
        'categoryId':       int(categoria),
        'duration_seconds': duracion_seg,
        'duration_minutes': duracion_seg / 60,
        'title_len':        len(titulo),
        'desc_len':         len(descripcion),
        'text_len':         len(texto),
        'title_word_count': len(titulo.split()),
        'desc_word_count':  len(descripcion.split()),
        'exclamation_count': texto.count('!'),
        'question_count':   texto.count('?'),
        'digit_count':      sum(c.isdigit() for c in texto),
        'uppercase_ratio':  sum(1 for c in titulo if c.isupper())/max(len(titulo),1),
        'viewCount':        viewcount if modo == 'postpublicacion' else 0,
        'log_viewCount':    np.log1p(viewcount) if modo == 'postpublicacion' else 0,
        'exhibitions':      exhibitions if modo == 'postpublicacion' else 1,
        'emotional_score':  min(texto.count('!')*0.5 + sum(c.isupper() for c in titulo)/max(len(titulo),1)*0.5, 1),
    })

    import re
    cta_kw = r'\b(agenda|reserva|llama|contacta|inscribete|descarga|visita|solicita|compra|suscribete)\b'
    trust_kw = r'\b(garantia|respaldo|certificado|oficial|confianza|verificado|premiado|profesional)\b'
    value_kw = r'\b(gratis|descuento|oferta|precio|ahorra|exclusivo|nuevo|mejor|garantia)\b'
    promo_kw = r'\b(solo hoy|no te pierdas|ultimas|ultimo|ya|ahora mismo|aprovecha)\b'
    row['cta_flag']              = int(bool(re.search(cta_kw, texto.lower())))
    row['trust_flag']            = int(bool(re.search(trust_kw, texto.lower())))
    row['value_proposition_flag'] = int(bool(re.search(value_kw, texto.lower())))
    row['promotional_flag']      = int(bool(re.search(promo_kw, texto.lower())))

    df_in = pd.DataFrame([row])
    for col in VALID_NUM:
        if col in df_in.columns:
            df_in[col] = pd.to_numeric(df_in[col], errors='coerce').fillna(0)

    proba = best_pipe.predict_proba(df_in)[0, 1]
    rec   = get_recomendacion(row, proba)

    return {'modo': modo, **rec}

# ── DEMO 1: Modo prepublicación ──────────────────────────────
print('=' * 60)
print('DEMO — MODO PREPUBLICACIÓN')
print('=' * 60)
r1 = analizar_contenido(
    titulo      = 'Apartamento de lujo en Quito con rooftop y vista panoramica',
    descripcion = 'Descubre un apartamento moderno. Agenda tu visita hoy.',
    duracion_seg = 48,
    categoria   = 22,
    modo        = 'prepublicacion'
)
print(f'Probabilidad alto rendimiento: {r1["proba_alto_rendimiento"]:.1%}')
print(f'Nivel: {r1["nivel"]}')
print(f'Recomendacion: {r1["titulo_recomendacion"]}')
print(f'Accion: {r1["accion"][:120]}...')
print(f'Factores: {r1["factores"]}')
print()

# ── DEMO 2: Modo postpublicación temprana ─────────────────────
print('=' * 60)
print('DEMO — MODO POSTPUBLICACION TEMPRANA')
print('=' * 60)
r2 = analizar_contenido(
    titulo      = 'Recorrido visual de proyecto residencial contemporaneo',
    descripcion = 'Narrativa institucional sin cierre comercial directo.',
    duracion_seg = 95,
    categoria   = 22,
    viewcount   = 19645,
    exhibitions = 1,
    modo        = 'postpublicacion'
)
print(f'Probabilidad alto rendimiento: {r2["proba_alto_rendimiento"]:.1%}')
print(f'Nivel: {r2["nivel"]}')
print(f'Recomendacion: {r2["titulo_recomendacion"]}')
print(f'Accion: {r2["accion"][:120]}...')
print(f'Factores: {r2["factores"]}')
print()

# ── Exportar resultado ────────────────────────────────────────
df_demo = pd.DataFrame([r1, r2])
df_demo.to_csv(RESULTS / 'demo_inferencia.csv', index=False)
print('Demo exportado a RESULTS/demo_inferencia.csv')

## Celda final — Resumen ejecutivo y artefactos generados

In [ ]:
# ============================================================
# Resumen ejecutivo de la ejecución completa.
# Consolida métricas, artefactos y hallazgos para la defensa.
# ============================================================
from datetime import datetime

best_row = df_metrics[df_metrics['modelo'] == best_name].iloc[0]

summary = {
    'fecha': datetime.now().isoformat(),
    'proyecto': 'Sistema Hibrido de IA Explicable para Clasificar Potencial de Rendimiento Publicitario',
    'dataset': {'registros': int(df.shape[0]), 'variables_originales': int(df_raw.shape[1])},
    'variable_objetivo': {
        'nombre': 'high_performance',
        'descripcion': 'Potencial relativo de rendimiento (score compuesto normalizado)',
        'prevalencia_alto': float(df['high_performance'].mean()),
        'p33': float(p33), 'p66': float(p66)
    },
    'features': {'numericas': len(VALID_NUM), 'categoricas': len(VALID_CAT), 'textual': 1},
    'clustering': {'k': K_OPTIMAL, 'varianza_pca': float(var_ret)},
    'mejor_modelo': {
        'nombre': best_name,
        'f1_cv':  round(cv_res[best_name]['f1_mean'], 4),
        'f1_test': round(float(best_row['f1']), 4),
        'roc_auc_test': round(float(best_row['roc_auc']), 4),
        'recall_test': round(float(best_row['recall']), 4),
        'precision_test': round(float(best_row['precision']), 4),
    },
    'llm': 'Phi-3-mini-4k-instruct (si GPU disponible) / reglas estructuradas (fallback)',
    'limite_explicito': [
        'No garantiza viralidad ni exito comercial',
        'No automatiza presupuesto ni toma decisiones irreversibles',
        'No afirma causalidad fuerte',
        'Requiere revision humana (Human-in-the-Loop)',
        'Rendimiento basado en corpus publico 2007-2014',
    ],
    'artefactos': [str(p) for p in sorted(RESULTS.rglob('*')) if p.is_file()]
}

with open(RESULTS / 'resumen_ejecucion.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('=' * 62)
print('  RESUMEN EJECUTIVO DEL SISTEMA')
print('=' * 62)
print(f'  Proyecto: {summary["proyecto"][:60]}...')
print(f'  Dataset: {summary["dataset"]["registros"]:,} anuncios')
print(f'  Variable objetivo: {summary["variable_objetivo"]["nombre"]}')
print(f'  Prevalencia alto rendimiento: {summary["variable_objetivo"]["prevalencia_alto"]:.2%}')
print()
print('  MODELO PRINCIPAL')
print(f'  Nombre:    {summary["mejor_modelo"]["nombre"]}')
print(f'  F1 (CV):   {summary["mejor_modelo"]["f1_cv"]}')
print(f'  F1 (test): {summary["mejor_modelo"]["f1_test"]}')
print(f'  ROC-AUC:   {summary["mejor_modelo"]["roc_auc_test"]}')
print(f'  Recall:    {summary["mejor_modelo"]["recall_test"]}')
print(f'  Precision: {summary["mejor_modelo"]["precision_test"]}')
print()
print('  ARTEFACTOS GENERADOS')
for a in summary['artefactos']:
    print(f'  {a}')
print()
print('  LÍMITES DECLARADOS')
for lim in summary['limite_explicito']:
    print(f'  - {lim}')
print()
print('  El sistema opera bajo Human-in-the-Loop.')
print('  El analista humano retiene la decision final.')
print()
print('Ejecucion completa.')